In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "cardiffnlp/twitter-roberta-base-emotion"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# zapis do folderu projektu
model.save_pretrained("./model")
tokenizer.save_pretrained("./model")

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

model = AutoModelForSequenceClassification.from_pretrained("./model")
tokenizer = AutoTokenizer.from_pretrained("./model")

text = "I feel really happy but also a bit nervous."
inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)
probs = F.softmax(outputs.logits, dim=1)
print(["anger", "joy", "optimism", "sadness"])
print(probs)

/home/chemik/miniconda3/envs/ggsn_ex2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['anger', 'joy', 'optimism', 'sadness']
tensor([[0.0114, 0.5687, 0.1162, 0.3038]], grad_fn=<SoftmaxBackward0>)


# Attention

In [7]:
outputs = model(**inputs, output_attentions=True)

attentions = outputs.attentions  # tuple warstw
last_layer = attentions[-1]      # ostatnia warstwa

# średnia po głowach
attn = last_layer.mean(dim=1).squeeze()

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

clean_tokens = [t.replace("Ġ", "") for t in tokens]

for token, score in zip(clean_tokens, attn[0]):
    print(token, float(score))

<s> 0.04959753528237343
I 0.1220237985253334
feel 0.11181914806365967
really 0.08374745398759842
happy 0.15168924629688263
but 0.06683406978845596
also 0.05248682573437691
a 0.05288331210613251
bit 0.0651257261633873
nervous 0.14460419118404388
. 0.04959233105182648
</s> 0.04959637299180031


# Lime

In [7]:
from lime.lime_text import LimeTextExplainer
import numpy as np

# POPRAW liczby klas
class_names = ["anger", "joy", "optimism", "sadness"]  # przykładowe – sprawdź model!

def predict_proba(texts):
    # 🔥 SHAP czasem daje pojedynczy string → zamień na listę
    if isinstance(texts, str):
        texts = [texts]

    inputs = tokenizer(
        list(texts),   # 🔥 WAŻNE
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)

    return probs.detach().numpy()

In [6]:
explainer = LimeTextExplainer(class_names=class_names)

exp = explainer.explain_instance(
    text,
    predict_proba,
    num_features=6,     # zmniejsz
    num_samples=100     # 🔥 KLUCZOWE – było ~5000
)

exp.as_list()

[('happy', 0.5535174534389021),
 ('nervous', -0.21787710984692787),
 ('feel', -0.12054295789805253),
 ('but', -0.0746113289509006),
 ('bit', 0.0533891989593408),
 ('I', -0.012743258734660857)]

# SHAP

In [9]:
import shap

explainer = shap.Explainer(predict_proba, masker=shap.maskers.Text(tokenizer), algorithm="partition")

shap_values = explainer([text])

shap.plots.text(shap_values[0])

# czerwone (negatywny wpływ) Słowa, które obniżają wynik danej klasy
# niebieskie (pozytywny wpływ) Słowa, które podbijają wynik klasy
# Output 0–3 To są Twoje klasy: anger, joy, optimism, sadness

# Gradients

In [10]:
from captum.attr import IntegratedGradients

# weź embedding layer
embeddings = model.roberta.embeddings

def forward_func(inputs_embeds, attention_mask):
    outputs = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
    return outputs.logits

# zamień input_ids → embeddings
input_embeds = embeddings.word_embeddings(input_ids)

ig = IntegratedGradients(forward_func)

attributions, delta = ig.attribute(
    inputs=input_embeds,
    additional_forward_args=(attention_mask,),
    target=torch.argmax(outputs.logits),
    return_convergence_delta=True
)

tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

clean_tokens = [t.replace("Ġ", "") for t in tokens]

for token, score in zip(clean_tokens, attributions[0]):
    print(token, float(score.sum().detach()))

<s> -0.13421323895454407
I -0.19204780459403992
feel -0.08344478905200958
really 0.4782661199569702
happy 1.693516492843628
but -0.07846126705408096
also 0.3125418424606323
a 0.051396824419498444
bit -0.11531603336334229
nervous -2.01340913772583
. 0.007139723747968674
</s> 1.6517846584320068
